#Ingest - Race Data & Race Events - _James Trotman_
> Gets all the race data & race events CSVs from s3 for Race Data

In [0]:
%run /Workspace/Repos/nikum.vedansh@gmail.com/formula-one-project/configs/credentials

In [0]:
dbutils.widgets.text("schema", "default", "Schema")

paths = [RACE_DATA_PATH, RACE_EVENTS_PATH]

succeeded = []
failed = []
skipped = []

In [0]:
# 1. Faster Catalog Check: Pre-fetch existing tables
current_schema = dbutils.widgets.get("schema")
full_schema_path = f"formone.{current_schema}"

In [0]:
import re
import time
from datetime import datetime
from multiprocessing.pool import ThreadPool
from pyspark.sql.functions import current_timestamp

In [0]:
try:
    # Get all tables in one go and store in a set (O(1) lookup time)
    existing_tables = {t.name for t in spark.catalog.listTables(full_schema_path)}
except:
    existing_tables = set()

# Configuration
max_parallel_tasks = 16
invalid_chars_pattern = re.compile(r"[ ,;{}()\n\t=]")

def sanitize_columns(df):
    new_cols = [invalid_chars_pattern.sub("_", c.lower()) for c in df.columns]
    return df.toDF(*new_cols).withColumn("_ingested_at", current_timestamp())

def process_single_file(file_info):
    path, folder_name = file_info
    file_name = path.split("/")[-1]
    
    if not file_name.endswith(".csv"):
        return ("skipped", file_name)

    clean_name = file_name.replace(".csv", "")
    table_ident = f"{folder_name}_{clean_name}"
    full_table_path = f"{full_schema_path}.{table_ident}"

    # Optimization: Local set lookup instead of spark.catalog call
    if table_ident in existing_tables:
        return ("skipped", file_name)

    try:
        # Optimization: Use multiline if your CSVs have quoted newlines, 
        # otherwise keep default for speed.
        df = spark.read.format("csv") \
            .option("header", "true") \
            .option("inferSchema", "false") \
            .load(path)
        
        df = sanitize_columns(df)
        
        # Optimization: Use Delta 'overwriteSchema' if you expect columns to change
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(full_table_path)
            
        return ("succeeded", file_name)
    except Exception as e:
        return ("failed", f"{file_name} (Error: {str(e)})")

# --- Timing and Execution ---
start_time = time.time()

# Flatten file list
all_files_to_process = []
for path in paths:
    # Using folder name from path logic
    folder_name = path.split("/")[-3]
    try:
        for f in dbutils.fs.ls(path):
            all_files_to_process.append((f.path, folder_name))
    except:
        continue

pool = ThreadPool(max_parallel_tasks)
results = pool.map(process_single_file, all_files_to_process)

total_duration = round(time.time() - start_time, 2)

# --- Summary Logic ---
summary = {"succeeded": [], "failed": [], "skipped": []}
for status, name in results:
    summary[status].append(name)

print(f"\n{'='*40}\nBronze Ingestion Report: {total_duration}s\n{'='*40}")
for key, icon in [("succeeded", "✅"), ("skipped", "⏭️"), ("failed", "❌")]:
    print(f"{icon} {key.capitalize()} ({len(summary[key])}):")
    for item in summary[key]: print(f"  - {item}")